# Latent Dirichlet Allocation & topic models — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normalize(v):
    v=np.asarray(v,dtype=float); return v/v.sum()
def softmax(v):
    v=np.asarray(v,dtype=float); e=np.exp(v-v.max()); return e/e.sum()
def norm_pdf(x,mu,var):
    return np.exp(-0.5*(np.asarray(x)-mu)**2/var)/np.sqrt(2*np.pi*var)
def beta_pdf_grid(x,a,b):
    B=math.gamma(a)*math.gamma(b)/math.gamma(a+b)
    return (np.asarray(x)**(a-1))*((1-np.asarray(x))**(b-1))/B
print('setup ok')

## Explain documents as mixtures of topics and topics as mixtures of words
LDA uses latent topics to share word patterns across documents. These examples compute document-topic means, topic-word means, a collapsed Gibbs conditional, a perplexity-style likelihood, and the effect of alpha.

In [ ]:
# 1) document-topic posterior mean from counts plus alpha
alpha=np.array([0.5,0.5]); nd=np.array([8,2]); theta=(nd+alpha)/(nd.sum()+alpha.sum())
plt.figure(figsize=(6,3)); plt.bar(['topic0','topic1'],theta); plt.title('document-topic mixture')
assert np.allclose(theta,[0.7727272727272727,0.22727272727272727])

In [ ]:
# 2) topic-word posterior mean from counts plus eta
eta=0.1; nkw=np.array([[9,1,1],[1,4,5]],dtype=float); phi=(nkw+eta)/(nkw.sum(1,keepdims=True)+3*eta)
plt.figure(figsize=(6,3)); plt.imshow(phi); plt.colorbar(); plt.title('topic-word distributions')
assert phi[0,0]>phi[1,0]

In [ ]:
# 3) collapsed Gibbs conditional is document term times word term
word=2; cond=(nd+alpha)*phi[:,word]; cond=cond/cond.sum()
plt.figure(figsize=(6,3)); plt.bar(['topic0','topic1'],cond); plt.title('topic assignment probability for word 2')
assert cond[1]>cond[0]

In [ ]:
# 4) document word probability is mixture theta dot phi
pw=theta@phi; plt.figure(figsize=(6,3)); plt.bar(['w0','w1','w2'],pw); plt.title('predictive word distribution')
assert abs(pw.sum()-1)<1e-12

In [ ]:
# 5) smaller alpha makes document mixtures sparser
alphas=[0.1,0.5,5.0]; spars=[]
for a in alphas:
    th=(nd+a)/(nd.sum()+2*a); spars.append(th.max())
plt.figure(figsize=(6,3)); plt.plot(alphas,spars,'-o'); plt.xscale('log'); plt.xlabel('alpha'); plt.ylabel('largest topic share'); plt.title('alpha controls mixture concentration')
assert spars[0]>spars[-1]